# Guía · Red neuronal desde cero: NumPy → PyTorch

> 🇬🇧 English version: [`01_neural_network_from_scratch_en.ipynb`](01_neural_network_from_scratch_en.ipynb)

En esta guía **tú** construyes un **perceptrón multicapa (MLP)** usando solo NumPy: activaciones, pérdida,
*forward*, **backpropagation derivada a mano** y descenso de gradiente. Después escribes la misma red en
PyTorch y compruebas que **sus gradientes coinciden con los tuyos**.

### Cómo usar esta guía
- Las celdas marcadas con ✍️ tienen huecos `# TODO`. Sustituye cada `raise NotImplementedError` por tu código.
- Justo después hay una celda ✅ que **comprueba** tu implementación. Si pasa, sigue adelante; si falla,
  el mensaje te dice qué revisar.
- La teoría de la sección 2 contiene todas las ecuaciones que necesitas. Intenta no buscar la solución:
  equivocarte con las formas de las matrices es parte del aprendizaje.

Objetivos:
1. Entender qué calcula cada capa y por qué hace falta una **no linealidad**.
2. Derivar y programar la **regla de la cadena** en forma matricial.
3. Verificar tu implementación con un **chequeo numérico de gradientes**.
4. Ver qué automatiza PyTorch (`autograd`, `nn.Module`, `optim`) y qué no.

In [ ]:
# Librerias a usar
#=====================================================
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)
print("numpy", np.__version__)

## 1. Datos: tres espirales

Un problema clásico que **ningún modelo lineal** puede resolver: tres clases enrolladas en espiral.
Esta celda ya está completa; ejecútala y observa los datos.

In [ ]:
def espirales(n_por_clase=200, n_clases=3, ruido=0.2, semilla=0):
    """Devuelve X de forma (n, 2) y etiquetas y de forma (n,) con valores 0..n_clases-1."""
    g = np.random.default_rng(semilla)
    X, y = [], []
    for k in range(n_clases):
        r = np.linspace(0.0, 1.0, n_por_clase)                       # radio
        t = np.linspace(k * 4, (k + 1) * 4, n_por_clase) + g.normal(0, ruido, n_por_clase)  # ángulo
        X.append(np.column_stack([r * np.sin(t), r * np.cos(t)]))
        y.append(np.full(n_por_clase, k))
    return np.vstack(X), np.concatenate(y)

X, y = espirales()

# División train / test (80 / 20)
idx = rng.permutation(len(X))
corte = int(0.8 * len(X))
X_train, y_train = X[idx[:corte]], y[idx[:corte]]
X_test, y_test = X[idx[corte:]], y[idx[corte:]]
print(X_train.shape, X_test.shape, np.bincount(y_train))

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=12)
plt.title("Tres espirales"); plt.axis("equal"); plt.show()

## 2. La matemática

Para un *batch* $X \in \mathbb{R}^{N \times D}$ y una red con capas $\ell = 1, \dots, L$:

$$
Z^{(\ell)} = A^{(\ell-1)} W^{(\ell)} + b^{(\ell)}, \qquad
A^{(\ell)} = \mathrm{ReLU}\big(Z^{(\ell)}\big), \qquad A^{(0)} = X
$$

La última capa no lleva ReLU: produce **logits** $Z^{(L)} \in \mathbb{R}^{N \times K}$, que convertimos en
probabilidades con **softmax** y comparamos con la clase real mediante la **entropía cruzada**:

$$
P_{ik} = \frac{e^{Z_{ik}}}{\sum_j e^{Z_{ij}}}, \qquad
\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log P_{i, y_i}
$$

### Backpropagation (regla de la cadena en forma matricial)

Sea $Y$ la matriz *one-hot* de etiquetas. La combinación softmax + entropía cruzada tiene un gradiente
sorprendentemente simple:

$$
\frac{\partial \mathcal{L}}{\partial Z^{(L)}} = \frac{1}{N}\,(P - Y)
$$

A partir de ahí, para cada capa (de la última a la primera), con $\delta^{(\ell)} = \partial \mathcal{L} / \partial Z^{(\ell)}$:

$$
\frac{\partial \mathcal{L}}{\partial W^{(\ell)}} = A^{(\ell-1)\top} \delta^{(\ell)}, \qquad
\frac{\partial \mathcal{L}}{\partial b^{(\ell)}} = \sum_{i} \delta^{(\ell)}_{i,:}, \qquad
\delta^{(\ell-1)} = \big(\delta^{(\ell)} W^{(\ell)\top}\big) \odot \mathbb{1}\big[Z^{(\ell-1)} > 0\big]
$$

$\odot$ es el producto elemento a elemento y $\mathbb{1}[Z>0]$ es la derivada de ReLU.
Fíjate en que **las formas cuadran**: $A^{(\ell-1)\top}$ es $(n_{\ell-1} \times N)$ y $\delta^{(\ell)}$ es
$(N \times n_\ell)$, así que el producto tiene la forma de $W^{(\ell)}$. Comprobar formas es la mejor
forma de depurar backprop.

> 💡 Antes de programar, intenta derivar $\partial \mathcal{L} / \partial Z^{(L)} = (P - Y)/N$ en papel.

## 3. Los bloques en NumPy

### 3.1 ✍️ Activación, softmax y pérdida
- `relu`: $\max(0, z)$ elemento a elemento.
- `softmax`: resta el máximo de cada fila antes de `exp` (**softmax estable**). El resultado es idéntico,
  pero evitas desbordamientos con logits grandes. Pista: `keepdims=True`.
- `entropia_cruzada`: devuelve la pérdida media **y** su gradiente respecto a los logits, $(P - Y)/N$.
  Pista: `P[np.arange(n), y]` selecciona la probabilidad de la clase correcta de cada fila.

In [ ]:
def relu(z):
    # TODO: max(0, z) elemento a elemento
    raise NotImplementedError

def softmax(z):
    """Softmax por filas. z tiene forma (N, K)."""
    # TODO: resta el máximo de cada fila, aplica exp y normaliza para que cada fila sume 1
    raise NotImplementedError

def entropia_cruzada(logits, y):
    """Pérdida media y su gradiente respecto a los logits: (P - Y) / N."""
    n = len(y)
    P = softmax(logits)
    # TODO 1: perdida = media de -log(probabilidad de la clase correcta)  (suma 1e-12 dentro del log)
    # TODO 2: dlogits = (P - Y) / n, sin construir Y: copia P y resta 1 en la columna correcta de cada fila
    raise NotImplementedError

In [ ]:
# ✅ Comprobación 3.1
assert np.array_equal(relu(np.array([-2.0, 0.0, 3.0])), [0.0, 0.0, 3.0]), "relu: revisa los negativos"
p = softmax(np.array([[1.0, 2.0, 3.0], [1000.0, 1000.0, 1000.0]]))
assert np.all(np.isfinite(p)), "softmax: hay overflow, ¿restaste el máximo de cada fila?"
assert np.allclose(p.sum(axis=1), 1.0), "softmax: cada fila debe sumar 1"
assert np.allclose(p[0], [0.09003057, 0.24472847, 0.66524096]), "softmax: valores incorrectos"
perdida, d = entropia_cruzada(np.array([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3]]), np.array([0, 1]))
assert np.isclose(perdida, 0.3185, atol=1e-3), f"entropía cruzada: se esperaba ≈0.3185, obtuviste {perdida:.4f}"
assert np.allclose(d.sum(axis=1), 0.0), "gradiente: cada fila de (P - Y) debe sumar 0"
assert np.isclose(d[0, 0], (softmax(np.array([[2.0, 1.0, 0.1]]))[0, 0] - 1) / 2), "gradiente: ¿dividiste entre N?"
print("✅ 3.1 correcto")

### 3.2 ✍️ La red
Una clase con tres métodos que completas tú:
- `forward` guarda lo que backprop necesitará: las entradas de cada capa en `self.A` y las pre-activaciones
  en `self.Z`. La última capa **no** aplica ReLU.
- `backward` aplica las ecuaciones de la sección 2, de la última capa a la primera.
- `paso` actualiza los parámetros con descenso de gradiente: $\theta \leftarrow \theta - \eta\,\nabla_\theta \mathcal{L}$.

La inicialización ya está hecha con **He** ($\sigma = \sqrt{2 / n_{\text{entrada}}}$), pensada para ReLU: mantiene
la varianza de las activaciones aproximadamente constante entre capas. Si inicializas todo a cero, todas las
neuronas de una capa reciben el mismo gradiente y nunca se diferencian (**simetría**).

In [ ]:
class RedNeuronal:
    """MLP totalmente conectado: ReLU en capas ocultas, logits a la salida."""

    def __init__(self, tamanos, semilla=0):
        g = np.random.default_rng(semilla)
        self.W = [g.normal(0.0, np.sqrt(2.0 / n_ent), (n_ent, n_sal))
                  for n_ent, n_sal in zip(tamanos[:-1], tamanos[1:])]
        self.b = [np.zeros((1, n_sal)) for n_sal in tamanos[1:]]

    def forward(self, X):
        self.A, self.Z = [X], []
        for capa, (W, b) in enumerate(zip(self.W, self.b)):
            # TODO: calcula Z con la entrada self.A[-1], guárdala en self.Z y,
            #       si no es la última capa, guarda relu(Z) en self.A
            raise NotImplementedError
        return Z

    def backward(self, dZ):
        """Recibe dL/dlogits y devuelve los gradientes (dW, db) de cada capa."""
        dW, db = [None] * len(self.W), [None] * len(self.b)
        for capa in reversed(range(len(self.W))):
            # TODO 1: dW[capa] y db[capa] a partir de dZ y self.A[capa]  (db con keepdims=True)
            # TODO 2: si capa > 0, propaga dZ a la capa anterior (usa self.W[capa] y self.Z[capa - 1])
            raise NotImplementedError
        return dW, db

    def paso(self, dW, db, lr):
        # TODO: descenso de gradiente sobre cada W y b
        raise NotImplementedError

    def predecir(self, X):
        return self.forward(X).argmax(axis=1)

In [ ]:
# ✅ Comprobación 3.2 (formas)
red = RedNeuronal([2, 64, 64, 3])
logits = red.forward(X_train[:10])
assert logits.shape == (10, 3), f"forward: los logits deben ser (10, 3), no {logits.shape}"
assert len(red.A) == 3 and len(red.Z) == 3, "forward: guarda 3 entradas en self.A y 3 pre-activaciones en self.Z"
dW, db = red.backward(np.ones((10, 3)))
assert [g.shape for g in dW] == [W.shape for W in red.W], "backward: cada dW debe tener la forma de su W"
assert [g.shape for g in db] == [b.shape for b in red.b], "backward: cada db debe tener la forma de su b"
print("Parámetros:", sum(W.size + b.size for W, b in zip(red.W, red.b)))
print("✅ 3.2 formas correctas (la sección 4 comprueba los valores)")

## 4. ✅ Chequeo numérico de gradientes

Antes de entrenar, **verificamos** tu `backward` comparándolo con diferencias finitas centradas:

$$
\frac{\partial \mathcal{L}}{\partial \theta} \approx \frac{\mathcal{L}(\theta + \varepsilon) - \mathcal{L}(\theta - \varepsilon)}{2\varepsilon}
$$

Se usa una red pequeña y `float64`. Un error relativo de $10^{-7}$ o menor indica que tu derivación es
correcta; algo como $10^{-2}$ indica un error en backprop. Esta celda ya está completa: léela con calma,
es una herramienta que usarás siempre que programes un gradiente a mano.

In [ ]:
def chequeo_gradientes(red, X, y, eps=1e-5):
    perdida, dlogits = entropia_cruzada(red.forward(X), y)
    dW, db = red.backward(dlogits)
    errores = []
    for params, grads in ((red.W, dW), (red.b, db)):
        for P, G in zip(params, grads):
            num = np.zeros_like(P)
            for i in np.ndindex(P.shape):
                original = P[i]
                P[i] = original + eps; mas, _ = entropia_cruzada(red.forward(X), y)
                P[i] = original - eps; menos, _ = entropia_cruzada(red.forward(X), y)
                P[i] = original
                num[i] = (mas - menos) / (2 * eps)
            errores.append(np.abs(num - G).max() / max(np.abs(num).max() + np.abs(G).max(), 1e-12))
    return max(errores)

error = chequeo_gradientes(RedNeuronal([2, 5, 4, 3], semilla=1), X_train[:20], y_train[:20])
print(f"Error relativo máximo: {error:.2e}")
assert error < 1e-7, "backward no coincide con el gradiente numérico: revisa las transpuestas y la derivada de ReLU"
print("✅ 4 tu backpropagation es correcta")

## 5. ✍️ Entrenamiento con mini-batches

El ciclo de épocas y el barajado ya están escritos. Tú completas **un paso de entrenamiento** sobre un
mini-batch: **forward → pérdida → backward → paso**. Son exactamente los pasos que hace PyTorch por dentro.

Con `[2, 64, 64, 3]`, `lr=0.2` y 300 épocas deberías superar el **95 % de precisión en test**.

In [ ]:
def paso_entrenamiento(red, Xb, yb, lr):
    """Un paso de descenso de gradiente sobre el mini-batch (Xb, yb). Devuelve la pérdida."""
    # TODO: forward → entropia_cruzada → backward → paso. Devuelve la pérdida.
    raise NotImplementedError

def precision(red, X, y):
    return (red.predecir(X) == y).mean()

def entrenar(red, X, y, epocas=300, lr=0.2, tam_batch=32, semilla=0):
    g = np.random.default_rng(semilla)
    historial = []
    for epoca in range(epocas):
        orden = g.permutation(len(X))
        total = 0.0
        for inicio in range(0, len(X), tam_batch):
            lote = orden[inicio:inicio + tam_batch]
            total += paso_entrenamiento(red, X[lote], y[lote], lr) * len(lote)
        historial.append(total / len(X))
        if epoca % 50 == 0 or epoca == epocas - 1:
            print(f"época {epoca:3d} | pérdida {historial[-1]:.4f} | precisión train {precision(red, X, y):.3f}")
    return historial

red = RedNeuronal([2, 64, 64, 3], semilla=0)
historial_np = entrenar(red, X_train, y_train)
prec_np = precision(red, X_test, y_test)
print(f"\nPrecisión en test: {prec_np:.3f}")

plt.plot(historial_np)
plt.yscale("log"); plt.xlabel("época"); plt.ylabel("entropía cruzada"); plt.grid(True)
plt.title("Curva de aprendizaje (NumPy)"); plt.show()

In [ ]:
# ✅ Comprobación 5
assert historial_np[-1] < historial_np[0] / 5, "la pérdida apenas baja: revisa el signo de la actualización en paso()"
assert prec_np > 0.95, f"precisión en test {prec_np:.3f}: debería superar 0.95"
print("✅ 5 tu red aprende las espirales")

### 5.1 Fronteras de decisión: ¿por qué hace falta la capa oculta?

Una red **sin capas ocultas** (`[2, 3]`) es una regresión logística multinomial: solo puede trazar fronteras
**rectas**. Al apilar capas con ReLU, la red compone muchas regiones lineales y puede aproximar fronteras
tan curvas como las espirales. Ejecuta la celda y compara.

In [ ]:
def dibujar_frontera(predecir, X, y, titulo, ax):
    xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 300), np.linspace(-1.2, 1.2, 300))
    malla = np.column_stack([xx.ravel(), yy.ravel()])
    ax.contourf(xx, yy, predecir(malla).reshape(xx.shape), alpha=0.3, cmap="viridis")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=10)
    ax.set_title(titulo); ax.set_aspect("equal")

lineal = RedNeuronal([2, 3], semilla=0)
_ = entrenar(lineal, X_train, y_train, epocas=100, lr=0.2)

fig, ejes = plt.subplots(1, 2, figsize=(11, 5))
dibujar_frontera(lineal.predecir, X_test, y_test,
                 f"Lineal [2, 3] · test {precision(lineal, X_test, y_test):.2f}", ejes[0])
dibujar_frontera(red.predecir, X_test, y_test,
                 f"MLP [2, 64, 64, 3] · test {precision(red, X_test, y_test):.2f}", ejes[1])
plt.show()

## 6. La misma red en PyTorch

Ahora reescribes todo con PyTorch. Lo que **desaparece** es justo lo que más trabajo costó:
`backward` (lo hace `autograd`), la inicialización (`nn.Linear`), la pérdida (`nn.CrossEntropyLoss`, que ya
incluye softmax) y la actualización (`torch.optim.SGD`).

### 6.1 ✍️ Construir el modelo
Completa `mlp_torch` para que, dada una lista de tamaños como `[2, 64, 64, 3]`, devuelva un `nn.Sequential`
con `nn.Linear` + `nn.ReLU` en cada capa, **sin** ReLU tras la última.

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)

def mlp_torch(tamanos):
    # TODO: construye la lista de capas y devuelve nn.Sequential(*capas)
    raise NotImplementedError

modelo = mlp_torch([2, 64, 64, 3])
print(modelo)

# ✅ Comprobación 6.1
assert [type(m).__name__ for m in modelo] == ["Linear", "ReLU", "Linear", "ReLU", "Linear"], \
    "el modelo debe ser Linear, ReLU, Linear, ReLU, Linear"
print("✅ 6.1 correcto")

### 6.2 ✅ ¿Coinciden tus gradientes con los de PyTorch?

Esta celda copia los pesos de tu red NumPy en el modelo de PyTorch y compara los gradientes de `autograd`
con los de **tu** `backward`. Ojo: `nn.Linear` guarda los pesos como $(n_{\text{sal}} \times n_{\text{ent}})$
y calcula $x W^\top + b$, por eso se **transpone**. Se trabaja en `float64` para comparar con precisión.

Si todo está bien, verás diferencias de $10^{-16}$ o menores: error de redondeo de la máquina.

In [ ]:
red_ref = RedNeuronal([2, 64, 64, 3], semilla=7)
gemelo = mlp_torch([2, 64, 64, 3]).double()
lineales = [m for m in gemelo if isinstance(m, nn.Linear)]
with torch.no_grad():
    for capa, W, b in zip(lineales, red_ref.W, red_ref.b):
        capa.weight.copy_(torch.from_numpy(W.T))
        capa.bias.copy_(torch.from_numpy(b.ravel()))

Xb, yb = X_train[:64], y_train[:64]

# NumPy: tu backward
perdida_np, dlogits = entropia_cruzada(red_ref.forward(Xb), yb)
dW, db = red_ref.backward(dlogits)

# PyTorch: autograd
perdida_t = nn.functional.cross_entropy(gemelo(torch.from_numpy(Xb)), torch.from_numpy(yb))
perdida_t.backward()

print(f"pérdida NumPy {perdida_np:.8f} | PyTorch {perdida_t.item():.8f}")
for i, capa in enumerate(lineales):
    dif = max(np.abs(capa.weight.grad.numpy().T - dW[i]).max(),
              np.abs(capa.bias.grad.numpy() - db[i].ravel()).max())
    print(f"capa {i}: diferencia máxima de gradiente = {dif:.1e}")
    assert dif < 1e-10, f"capa {i}: tus gradientes no coinciden con autograd"
print("✅ 6.2 tu backward calcula lo mismo que autograd")

### 6.3 ✍️ Entrenamiento en PyTorch
Mismos hiperparámetros que en NumPy (SGD, `lr=0.2`, batches de 32, 300 épocas). Completa el paso de
entrenamiento con los cuatro pasos que repetirás en cualquier proyecto de PyTorch:
`zero_grad` → `forward` + `loss` → `backward` → `step`.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

Xtr_t = torch.tensor(X_train, dtype=torch.float32); ytr_t = torch.tensor(y_train)
Xte_t = torch.tensor(X_test, dtype=torch.float32);  yte_t = torch.tensor(y_test)
cargador = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=32, shuffle=True)

modelo = mlp_torch([2, 64, 64, 3])
perdida_fn = nn.CrossEntropyLoss()
optimizador = torch.optim.SGD(modelo.parameters(), lr=0.2)

def paso_torch(xb, yb):
    """Un paso de entrenamiento en PyTorch. Devuelve la pérdida como tensor."""
    # TODO: los cuatro pasos de PyTorch con modelo, perdida_fn y optimizador
    raise NotImplementedError

historial_t = []
for epoca in range(300):
    modelo.train()
    total = 0.0
    for xb, yb in cargador:
        total += paso_torch(xb, yb).item() * len(xb)
    historial_t.append(total / len(cargador.dataset))

modelo.eval()
with torch.no_grad():
    prec_t = (modelo(Xte_t).argmax(dim=1) == yte_t).float().mean().item()
print(f"Precisión en test — NumPy: {prec_np:.3f} | PyTorch: {prec_t:.3f}")

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
ejes[0].plot(historial_np, label="NumPy"); ejes[0].plot(historial_t, label="PyTorch")
ejes[0].set_yscale("log"); ejes[0].set_xlabel("época"); ejes[0].set_ylabel("entropía cruzada")
ejes[0].legend(); ejes[0].grid(True)

def predecir_torch(M):
    with torch.no_grad():
        return modelo(torch.tensor(M, dtype=torch.float32)).argmax(dim=1).numpy()

dibujar_frontera(predecir_torch, X_test, y_test, f"PyTorch · test {prec_t:.2f}", ejes[1])
plt.show()

# ✅ Comprobación 6.3
assert prec_t > 0.95, f"precisión en test {prec_t:.3f}: debería superar 0.95"
print("✅ 6.3 tu red de PyTorch aprende las espirales")

Las curvas no serán idénticas porque la inicialización y el orden de los batches son distintos
(PyTorch usa una inicialización uniforme de Kaiming, no la normal de He), pero ambas redes deberían
llegar a un resultado parecido.

## Resumen
| Paso | NumPy (desde cero) | PyTorch |
|---|---|---|
| Parámetros | listas `W`, `b` con inicialización He | `nn.Linear` dentro de `nn.Sequential` |
| Forward | `A @ W + b`, `relu` | `modelo(x)` |
| Pérdida | `softmax` estable + entropía cruzada | `nn.CrossEntropyLoss` (recibe logits) |
| Backward | regla de la cadena derivada a mano | `perdida.backward()` (autograd) |
| Actualización | `W -= lr * dW` | `optimizador.step()` |
| Verificación | chequeo numérico de gradientes | — (confiamos en autograd) |

Programar backprop una vez a mano es lo que permite **depurar** redes reales: cuando una pérdida no baja,
sabes qué buscar (formas, signos, escalas de gradiente, inicialización).

## ✏️ Ejercicios extra
1. **Tanh**: cambia ReLU por $\tanh$ en tu red (su derivada es $1 - \tanh^2$). Repite el chequeo de
   gradientes y compara la velocidad de convergencia.
2. **Regularización L2**: añade $\frac{\lambda}{2}\sum \lVert W \rVert^2$ a la pérdida y $\lambda W$ a cada `dW`.
   Pruébalo con el chequeo numérico y observa cómo se suaviza la frontera con $\lambda = 10^{-3}$ y $10^{-2}$.
3. **Momentum y Adam**: implementa ambos optimizadores en NumPy y compara las curvas con SGD.
   Comprueba que coinciden con `torch.optim.SGD(momentum=0.9)` y `torch.optim.Adam`.
4. **Capacidad**: ¿cuál es la red más pequeña (ancho y profundidad) que supera el 95 % en test?
   ¿Qué pasa con 1 000 neuronas y pocos datos (`n_por_clase=30`)?
5. **Inicialización**: entrena con todos los pesos a cero y con $\sigma = 1$. Explica lo que observas.
6. **Datos reales**: aplica tu `RedNeuronal` a `sklearn.datasets.load_digits` (imágenes 8×8, 10 clases).
   Recuerda estandarizar las entradas.